In [ ]:
if len(df_loud) > 0:
    # Export loudest sources to CSV
    csv_file = RESULTS_PATH / 'loudest_cgw_candidates.csv'
    df_loud.to_csv(csv_file, index=False)
    print(f"✓ Exported loudest CGW candidates to: {csv_file}")
    
    # Export statistics to JSON
    stats_file = RESULTS_PATH / 'cgw_statistics.json'
    with open(stats_file, 'w') as f:
        json.dump(stats, f, indent=2)
    print(f"✓ Exported statistics to: {stats_file}")
    
    # Create summary report
    summary_text = f"""
================================================================================
SMBHB POPULATION CGW ANALYSIS - SUMMARY REPORT
================================================================================

Analysis Date: {datetime.now().isoformat()}
Results Directory: {RESULTS_PATH}

PIPELINE CONFIGURATION
  Populations analyzed: {stats['n_populations']}
  
LOUDEST CGW CANDIDATE STATISTICS
  
  CGW SNR:
    Mean:   {stats['cgw_snr']['mean']:.6f}
    Median: {stats['cgw_snr']['median']:.6f}
    Std:    {stats['cgw_snr']['std']:.6f}
    Range:  [{stats['cgw_snr']['min']:.6f}, {stats['cgw_snr']['max']:.6f}]
    IQR:    [{stats['cgw_snr']['p25']:.6f}, {stats['cgw_snr']['p75']:.6f}]

  Population sizes (N binaries):
    Mean:   {stats['n_binaries']['mean']:.0f}
    Median: {stats['n_binaries']['median']:.0f}
    Std:    {stats['n_binaries']['std']:.0f}
    Range:  [{stats['n_binaries']['min']:.0f}, {stats['n_binaries']['max']:.0f}]

  Frequencies:
    Mean:   {stats['frequency']['mean']:.2e} Hz
    Median: {stats['frequency']['median']:.2e} Hz
    Range:  [{stats['frequency']['min']:.2e}, {stats['frequency']['max']:.2e}] Hz

  Chirp masses:
    Mean:   {stats['chirp_mass']['mean']:.2e} kg
    Median: {stats['chirp_mass']['median']:.2e} kg
    Range:  [{stats['chirp_mass']['min']:.2e}, {stats['chirp_mass']['max']:.2e}] kg

EXPORTED FILES
  - loudest_cgw_candidates.csv     : Full candidate data (CSV)
  - cgw_statistics.json             : Summary statistics (JSON)
  - cgw_distributions.pdf           : Distribution plots (PDF)
  - cgw_distributions.png           : Distribution plots (PNG)
  - correlation_matrix.pdf          : Correlation heatmap (PDF)
  - correlation_matrix.png          : Correlation heatmap (PNG)

USAGE
  Load results in future analyses:
    import pandas as pd
    df = pd.read_csv('{csv_file}')
    
    import json
    with open('{stats_file}') as f:
        stats = json.load(f)

================================================================================
"""
    
    summary_file = RESULTS_PATH / 'ANALYSIS_SUMMARY.txt'
    with open(summary_file, 'w') as f:
        f.write(summary_text)
    
    print(summary_text)
    print(f"✓ Summary saved to: {summary_file}")
else:
    print("No results to export")

## 6. Export Results and Summary

In [ ]:
if len(df_loud) > 1:
    # Compute correlations
    cols_for_corr = ['cgw_snr', 'n_binaries', 'f', 'Mc', 'h0', 'D_comov', 'z']
    corr_matrix = df_loud[cols_for_corr].corr()
    
    # Plot correlation heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, 
                square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax,
                vmin=-1, vmax=1)
    ax.set_title('Correlation Matrix: Loudest CGW Properties', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_PATH / 'correlation_matrix.pdf', dpi=300, bbox_inches='tight')
    plt.savefig(RESULTS_PATH / 'correlation_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("Correlation Matrix:")
    print(corr_matrix)
    
    # Find strongest correlations
    print("\nStrongest correlations with CGW SNR:")
    cgw_corr = corr_matrix['cgw_snr'].drop('cgw_snr').abs().sort_values(ascending=False)
    for col, val in cgw_corr.items():
        print(f"  {col:15s}: {val:.4f}")
else:
    print("Need at least 2 populations for correlation analysis")

## 5. Correlation Analysis

In [ ]:
if len(df_loud) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: CGW SNR histogram
    axes[0, 0].hist(df_loud['cgw_snr'], bins=20, alpha=0.7, color='blue', edgecolor='black')
    axes[0, 0].axvline(stats['cgw_snr']['mean'], color='red', linestyle='--', linewidth=2, label='Mean')
    axes[0, 0].axvline(stats['cgw_snr']['median'], color='green', linestyle='--', linewidth=2, label='Median')
    axes[0, 0].set_xlabel('Loudest CGW SNR', fontsize=11)
    axes[0, 0].set_ylabel('Count', fontsize=11)
    axes[0, 0].set_title('Distribution of Loudest CGW SNR per Population', fontsize=12, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: CGW SNR vs N_binaries
    axes[0, 1].scatter(df_loud['n_binaries'], df_loud['cgw_snr'], alpha=0.6, s=100, color='purple')
    axes[0, 1].set_xlabel('N Binaries', fontsize=11)
    axes[0, 1].set_ylabel('Loudest CGW SNR', fontsize=11)
    axes[0, 1].set_title('CGW SNR vs Population Size', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Frequency histogram
    axes[1, 0].hist(df_loud['f'], bins=15, alpha=0.7, color='orange', edgecolor='black')
    axes[1, 0].set_xlabel('Frequency (Hz)', fontsize=11)
    axes[1, 0].set_ylabel('Count', fontsize=11)
    axes[1, 0].set_title('Frequency Distribution of Loudest Sources', fontsize=12, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: Chirp Mass histogram
    axes[1, 1].hist(df_loud['Mc'], bins=15, alpha=0.7, color='green', edgecolor='black')
    axes[1, 1].set_xlabel('Chirp Mass (kg)', fontsize=11)
    axes[1, 1].set_ylabel('Count', fontsize=11)
    axes[1, 1].set_title('Chirp Mass Distribution of Loudest Sources', fontsize=12, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(RESULTS_PATH / 'cgw_distributions.pdf', dpi=300, bbox_inches='tight')
    plt.savefig(RESULTS_PATH / 'cgw_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved distribution plots to {RESULTS_PATH}")
else:
    print("No data to visualize")

## 4. Visualization: CGW SNR Distributions

In [ ]:
if len(df_loud) > 0:
    # Compute summary statistics
    stats = {
        'n_populations': len(df_loud),
        'cgw_snr': {
            'mean': float(df_loud['cgw_snr'].mean()),
            'median': float(df_loud['cgw_snr'].median()),
            'std': float(df_loud['cgw_snr'].std()),
            'min': float(df_loud['cgw_snr'].min()),
            'max': float(df_loud['cgw_snr'].max()),
            'p25': float(df_loud['cgw_snr'].quantile(0.25)),
            'p75': float(df_loud['cgw_snr'].quantile(0.75)),
        },
        'n_binaries': {
            'mean': float(df_loud['n_binaries'].mean()),
            'median': float(df_loud['n_binaries'].median()),
            'std': float(df_loud['n_binaries'].std()),
            'min': float(df_loud['n_binaries'].min()),
            'max': float(df_loud['n_binaries'].max()),
        },
        'frequency': {
            'mean': float(df_loud['f'].mean()),
            'median': float(df_loud['f'].median()),
            'std': float(df_loud['f'].std()),
            'min': float(df_loud['f'].min()),
            'max': float(df_loud['f'].max()),
        },
        'chirp_mass': {
            'mean': float(df_loud['Mc'].mean()),
            'median': float(df_loud['Mc'].median()),
            'std': float(df_loud['Mc'].std()),
            'min': float(df_loud['Mc'].min()),
            'max': float(df_loud['Mc'].max()),
        },
    }
    
    # Print statistics
    print("="*70)
    print("LOUDEST CGW CANDIDATE STATISTICS")
    print("="*70)
    print(f"\nPopulations analyzed: {stats['n_populations']}")
    print(f"\nLoudest CGW SNR per population:")
    print(f"  Mean:   {stats['cgw_snr']['mean']:.4f}")
    print(f"  Median: {stats['cgw_snr']['median']:.4f}")
    print(f"  Std:    {stats['cgw_snr']['std']:.4f}")
    print(f"  Range:  [{stats['cgw_snr']['min']:.4f}, {stats['cgw_snr']['max']:.4f}]")
    print(f"  IQR:    [{stats['cgw_snr']['p25']:.4f}, {stats['cgw_snr']['p75']:.4f}]")
    
    print(f"\nBinaries per population:")
    print(f"  Mean:   {stats['n_binaries']['mean']:.0f}")
    print(f"  Median: {stats['n_binaries']['median']:.0f}")
    print(f"  Range:  [{stats['n_binaries']['min']:.0f}, {stats['n_binaries']['max']:.0f}]")
    
    print(f"\nFrequency of loudest sources:")
    print(f"  Mean:   {stats['frequency']['mean']:.2e} Hz")
    print(f"  Median: {stats['frequency']['median']:.2e} Hz")
    print(f"  Range:  [{stats['frequency']['min']:.2e}, {stats['frequency']['max']:.2e}] Hz")
    
    print(f"\nChirp mass of loudest sources:")
    print(f"  Mean:   {stats['chirp_mass']['mean']:.2e} kg")
    print(f"  Median: {stats['chirp_mass']['median']:.2e} kg")
    print(f"  Range:  [{stats['chirp_mass']['min']:.2e}, {stats['chirp_mass']['max']:.2e}] kg")
    print("="*70)
else:
    print("No data to analyze")

## 3. Aggregate Statistics

In [ ]:
# Load loudest CGW candidates from each population
loudest_files = sorted(RESULTS_PATH.glob("loudest_cgw_pop*.json"))
loudest_data = []

for f in loudest_files:
    with open(f) as fh:
        data = json.load(fh)
        loudest_data.append(data)

print(f"Loaded {len(loudest_data)} population results")

# Create DataFrame from loudest data
if loudest_data:
    df_loud = pd.DataFrame([
        {
            'pop_idx': d['pop_idx'],
            'n_binaries': d['n_binaries'],
            'cgw_snr': d['loudest_cgw']['cgw_snr'],
            'f': d['loudest_cgw']['f'],
            'Mc': d['loudest_cgw']['Mc'],
            'h0': d['loudest_cgw']['h0'],
            'D_comov': d['loudest_cgw']['D_comov'],
            'z': d['loudest_cgw']['z'],
            'ra': d['loudest_cgw']['ra'],
            'dec': d['loudest_cgw']['dec'],
            'psi': d['loudest_cgw']['psi'],
            'iota': d['loudest_cgw']['iota'],
            'phi0': d['loudest_cgw']['phi0'],
        }
        for d in loudest_data
    ])
    
    print(f"\n✓ Created DataFrame with {len(df_loud)} populations")
    print(df_loud.head(10))
else:
    print("No loudest CGW data found")

In [ ]:
# Specify results directory from HPC pipeline
# Update this path to point to your slurm chunks output directory
RESULTS_DIR = "data/chunks"  # Or: "data/2026-05-12/pessimistic_pipeline/chunks"

# Alternative: auto-discover latest results
if not Path(RESULTS_DIR).exists():
    # Find latest results directory
    data_dir = Path("data")
    if data_dir.exists():
        latest = sorted(data_dir.glob("*/*/chunks"))
        if latest:
            RESULTS_DIR = str(latest[-1])
            print(f"Auto-discovered results directory: {RESULTS_DIR}")

RESULTS_PATH = Path(RESULTS_DIR)

if not RESULTS_PATH.exists():
    print(f"ERROR: Results directory not found: {RESULTS_DIR}")
    print(f"Please run HPC pipeline first or update RESULTS_DIR path")
else:
    print(f"✓ Results directory: {RESULTS_PATH}")
    
    # List available results
    loudest_files = list(RESULTS_PATH.glob("loudest_cgw_pop*.json"))
    aggregated_file = RESULTS_PATH / "final_aggregated_results.json"
    
    print(f"  Found {len(loudest_files)} population loudest results")
    if aggregated_file.exists():
        print(f"  Found aggregated results file ✓")
    else:
        print(f"  Aggregated results file not found (will be created)")


## 2. Load Aggregated Results from HPC Pipeline

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import tracemalloc
import time
from typing import List, Dict, Any, Optional

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Configure output
np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 20)

print(f"Setup completed: {datetime.now().isoformat()}")

## 1. Setup and Configuration

# SMBHB Population CGW Analysis Pipeline

Comprehensive analysis of supermassive black hole binary populations with continuous gravitational wave (CGW) SNR analysis.

This notebook orchestrates:
1. Population generation with distance scaling to achieve target SNR
2. CGW SNR analysis for each population  
3. Extraction of "loudest" (highest CGW SNR) sources per population
4. Aggregate statistics across populations
5. Export of results for downstream analysis

**Key Outputs**:
- Per-population loudest CGW binary properties
- Aggregate statistics (CGW SNR distributions, correlations)
- Summary metadata for publication-ready analysis